# Восстановление пропущенных первых минут

Ноутбук для восстановления пропущенных первых минут в klines-файлах ADAUSDT.
Использует данные из trades для восстановления и перезаписывает файлы в S3.

In [1]:
import io
import os
from pathlib import Path
from typing import Iterable, Optional

import boto3
import pandas as pd
import yaml
from dotenv import load_dotenv

from restore_first_minute import restore_first_minute

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

load_dotenv()

True

In [2]:
with open("config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

DEFAULT_BUCKET = config["storage"]["bucket"]
DEFAULT_PREFIX = config["storage"]["prefix"]
DEFAULT_SOURCE = config["source"]
DEFAULT_SYMBOL = config["symbols"][0]

DEFAULT_BUCKET, DEFAULT_PREFIX, DEFAULT_SOURCE, DEFAULT_SYMBOL

('binance-data-downloader', 'raw_backup', 'trades', 'ADAUSDT')

In [3]:
def make_s3_client():
    return boto3.client(
        "s3",
        endpoint_url=os.getenv("YC_ENDPOINT"),
        region_name=os.getenv("YC_REGION"),
        aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
    )


s3 = make_s3_client()

In [4]:
def build_partition_prefix(
    base_prefix: str,
    source: str,
    symbol: Optional[str] = None,
    interval: Optional[str] = None,
    date_from: Optional[str] = None,
    date_to: Optional[str] = None,
) -> str:
    parts = [base_prefix.strip("/"), source]

    if symbol:
        parts.append(f"symbol={symbol}")

    if interval:
        parts.append(f"interval={interval}")

    prefix = "/".join(part for part in parts if part)

    if date_from and date_to and date_from == date_to:
        prefix = f"{prefix}/date={date_from}"

    return prefix.rstrip("/") + "/"


def _date_from_key(key: str) -> Optional[str]:
    for part in key.split("/"):
        if part.startswith("date="):
            return part.split("=", 1)[1]
    return None


def _read_parquet_from_s3(bucket: str, key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body))


def _write_parquet_to_s3(bucket: str, key: str, df: pd.DataFrame) -> None:
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    buffer.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())

In [5]:
# Загрузить список пропущенных дат
missing_dates_df = pd.read_csv("missing_dates_ada.csv", parse_dates=["date", "timestamp"])
missing_dates_df

,date,timestamp
0,2020-02-02,2020-02-02 00:00:00+00:00
1,2020-02-03,2020-02-03 00:00:00+00:00
2,2020-02-04,2020-02-04 00:00:00+00:00
3,2020-02-05,2020-02-05 00:00:00+00:00
4,2020-02-06,2020-02-06 00:00:00+00:00
...,...,...
893,2022-08-06,2022-08-06 00:00:00+00:00
894,2022-08-07,2022-08-07 00:00:00+00:00
895,2022-08-08,2022-08-08 00:00:00+00:00
896,2022-08-09,2022-08-09 00:00:00+00:00


In [6]:
def restore_missing_minute_for_date(target_date: str, dry_run: bool = False) -> bool:
    """
    Восстановить пропущенную первую минуту для указанной даты.
    
    Args:
        target_date: Дата в формате YYYY-MM-DD
        dry_run: Если True, только показать что будет сделано, без изменений
    
    Returns:
        True если восстановление успешно, False если пропустить
    """
    print(f"\n=== Processing {target_date} ===")
    
    # Найти ключи для klines и trades
    klines_prefix = build_partition_prefix(
        base_prefix=DEFAULT_PREFIX,
        source="klines",
        symbol=DEFAULT_SYMBOL,
        interval="1m",
        date_from=target_date,
        date_to=target_date,
    )
    
    trades_prefix = build_partition_prefix(
        base_prefix=DEFAULT_PREFIX,
        source="trades",
        symbol=DEFAULT_SYMBOL,
        date_from=target_date,
        date_to=target_date,
    )
    
    # Найти файлы
    klines_objects = s3.list_objects_v2(Bucket=DEFAULT_BUCKET, Prefix=klines_prefix)
    trades_objects = s3.list_objects_v2(Bucket=DEFAULT_BUCKET, Prefix=trades_prefix)
    
    klines_keys = [obj["Key"] for obj in klines_objects.get("Contents", []) if obj["Key"].endswith(".parquet")]
    trades_keys = [obj["Key"] for obj in trades_objects.get("Contents", []) if obj["Key"].endswith(".parquet")]
    
    if not klines_keys:
        print(f"❌ No klines file found for {target_date}")
        return False
    
    if not trades_keys:
        print(f"❌ No trades file found for {target_date}")
        return False
    
    klines_key = klines_keys[0]
    trades_key = trades_keys[0]
    
    print(f"Klines: s3://{DEFAULT_BUCKET}/{klines_key}")
    print(f"Trades: s3://{DEFAULT_BUCKET}/{trades_key}")
    
    # Скачать данные
    klines_df = _read_parquet_from_s3(DEFAULT_BUCKET, klines_key)
    trades_df = _read_parquet_from_s3(DEFAULT_BUCKET, trades_key)
    
    print(f"Klines shape: {klines_df.shape}")
    print(f"Trades shape: {trades_df.shape}")
    
    # Проверить, действительно ли пропущена первая минута
    if klines_df.empty:
        print("❌ Klines file is empty")
        return False
    
    first_timestamp = klines_df["timestamp"].min()
    expected_first_minute = pd.Timestamp(target_date + " 00:00:00+00:00")
    
    if first_timestamp == expected_first_minute:
        print(f"✓ First minute already present for {target_date}")
        return True
    
    print(f"Missing first minute: {expected_first_minute}")
    print(f"First available: {first_timestamp}")
    
    # Создать временные файлы для восстановления
    temp_dir = Path("temp_restore")
    temp_dir.mkdir(exist_ok=True)
    
    temp_klines = temp_dir / f"klines_{target_date}.parquet"
    temp_trades = temp_dir / f"trades_{target_date}.parquet"
    temp_restored = temp_dir / f"klines_restored_{target_date}.parquet"
    
    try:
        # Сохранить временные файлы
        klines_df.to_parquet(temp_klines, index=False)
        trades_df.to_parquet(temp_trades, index=False)
        
        # Восстановить минуту
        restored_df = restore_first_minute(temp_klines, temp_trades, temp_restored)
        
        print(f"Restored shape: {restored_df.shape}")
        print(f"New first timestamp: {restored_df['timestamp'].min()}")
        
        if dry_run:
            print("DRY RUN: Would upload restored file")
            return True
        
        # Перезаписать файл в S3
        _write_parquet_to_s3(DEFAULT_BUCKET, klines_key, restored_df)
        print(f"✓ Successfully restored and uploaded to s3://{DEFAULT_BUCKET}/{klines_key}")
        
        return True
        
    finally:
        # Очистить временные файлы (с повторными попытками из-за PyArrow)
        import time
        for temp_file in [temp_klines, temp_trades, temp_restored]:
            if temp_file.exists():
                for attempt in range(3):
                    try:
                        temp_file.unlink()
                        break
                    except PermissionError:
                        if attempt < 2:
                            time.sleep(0.1)  # Небольшая задержка
                        else:
                            print(f"Warning: Could not delete {temp_file}")
        
        if temp_dir.exists() and not list(temp_dir.iterdir()):
            try:
                temp_dir.rmdir()
            except PermissionError:
                pass

In [7]:
# Тест на одной дате (dry run)
test_date = missing_dates_df["date"].iloc[0].strftime("%Y-%m-%d")
print(f"Testing restoration for {test_date} (dry run)")
restore_missing_minute_for_date(test_date, dry_run=True)

Testing restoration for 2020-02-02 (dry run)

=== Processing 2020-02-02 ===
Klines: s3://binance-data-downloader/raw_backup/klines/symbol=ADAUSDT/interval=1m/date=2020-02-02/data.parquet
Trades: s3://binance-data-downloader/raw_backup/trades/symbol=ADAUSDT/date=2020-02-02/data.parquet
Klines shape: (1439, 12)
Trades shape: (49440, 7)
Missing first minute: 2020-02-02 00:00:00+00:00
First available: 2020-02-02 00:01:00+00:00
Restored shape: (1440, 12)
New first timestamp: 2020-02-02 00:00:00+00:00
DRY RUN: Would upload restored file


True

In [8]:
# Восстановить все пропущенные даты
# ⚠️  Это перезапишет файлы в S3! Убедитесь что создана резервная копия.

success_count = 0
total_count = len(missing_dates_df)

for idx, row in missing_dates_df.iterrows():
    target_date = row["date"].strftime("%Y-%m-%d")
    try:
        if restore_missing_minute_for_date(target_date, dry_run=False):
            success_count += 1
    except Exception as e:
        print(f"❌ Error processing {target_date}: {e}")
    
    print(f"Progress: {idx + 1}/{total_count} completed")

print(f"\n=== Summary ===")
print(f"Total dates processed: {total_count}")
print(f"Successfully restored: {success_count}")
print(f"Errors: {total_count - success_count}")


=== Processing 2020-02-02 ===
Klines: s3://binance-data-downloader/raw_backup/klines/symbol=ADAUSDT/interval=1m/date=2020-02-02/data.parquet
Trades: s3://binance-data-downloader/raw_backup/trades/symbol=ADAUSDT/date=2020-02-02/data.parquet
Klines shape: (1439, 12)
Trades shape: (49440, 7)
Missing first minute: 2020-02-02 00:00:00+00:00
First available: 2020-02-02 00:01:00+00:00
Restored shape: (1440, 12)
New first timestamp: 2020-02-02 00:00:00+00:00
✓ Successfully restored and uploaded to s3://binance-data-downloader/raw_backup/klines/symbol=ADAUSDT/interval=1m/date=2020-02-02/data.parquet
Progress: 1/898 completed

=== Processing 2020-02-03 ===
Klines: s3://binance-data-downloader/raw_backup/klines/symbol=ADAUSDT/interval=1m/date=2020-02-03/data.parquet
Trades: s3://binance-data-downloader/raw_backup/trades/symbol=ADAUSDT/date=2020-02-03/data.parquet
Klines shape: (1439, 12)
Trades shape: (48859, 7)
Missing first minute: 2020-02-03 00:00:00+00:00
First available: 2020-02-03 00:01:00